<a href="https://colab.research.google.com/github/divyankabhadauria-ui/CPRI-Hackathon-2026-The-Fitfth-Byte/blob/main/notebooks/FINAL_PIPELINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import pandas as pd
import numpy as np

from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestClassifier
)

from sklearn.model_selection import KFold, cross_val_score

INPUT_FILE = "CPRI_Hackathon_Screening_Dataset_PARTICIPANT.xlsx"

OUTPUT_FOLDER = "final_output"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

TEAM_NAME = "The Fifth Byte"

In [6]:
import pandas as pd

file = "CPRI_Hackathon_Screening_Dataset_PARTICIPANT.xlsx"

train = pd.read_excel(file, sheet_name="Training_Data")
test = pd.read_excel(file, sheet_name="Test_Data")

print("Training shape:", train.shape)
print("Test shape:", test.shape)

train.head()

Training shape: (1000, 11)
Test shape: (350, 9)


,Test_ID,Applied_Voltage_kV,Load_Current_A,Ambient_Temperature_C,Test_Duration_min,Sensor_S1,Sensor_S2,Sensor_S3,Sensor_S4,Reference_Parameter,Validity_Label
0,TRN-0889,16.7196,93.1228,33.3421,19.9546,13.6343,15.1361,16.5062,62.9115,34.8501,Valid
1,TRN-0820,21.4477,82.5230,34.4915,47.0553,15.6466,15.9849,19.3289,39.8667,30.4762,Valid
2,TRN-0411,19.2432,107.3619,29.4673,5.2763,15.1080,16.6481,18.3834,44.5390,46.8046,Valid
3,TRN-0754,22.7191,69.9690,26.9695,20.3051,14.9296,14.7026,18.6269,44.2673,23.7153,Valid
4,TRN-0707,13.5008,108.8999,35.5415,29.1939,12.6845,15.2455,14.6132,44.0720,48.5994,Valid


In [7]:
print("Missing values in training data:")
print(train.isnull().sum())

print("\nMissing values in test data:")
print(test.isnull().sum())

print("\nValidity label counts:")
print(train["Validity_Label"].value_counts())

print("\nDuplicate training rows:", train.duplicated().sum())
print("Duplicate test rows:", test.duplicated().sum())

print("\nDuplicate training Test_IDs:", train["Test_ID"].duplicated().sum())
print("Duplicate test Test_IDs:", test["Test_ID"].duplicated().sum())

Missing values in training data:
Test_ID                   0
Applied_Voltage_kV        0
Load_Current_A            0
Ambient_Temperature_C     0
Test_Duration_min         0
Sensor_S1                 6
Sensor_S2                 2
Sensor_S3                 7
Sensor_S4                29
Reference_Parameter       0
Validity_Label            0
dtype: int64

Missing values in test data:
Test_ID                   0
Applied_Voltage_kV        0
Load_Current_A            0
Ambient_Temperature_C     0
Test_Duration_min         0
Sensor_S1                 1
Sensor_S2                 3
Sensor_S3                 2
Sensor_S4                11
dtype: int64

Validity label counts:
Validity_Label
Valid      866
Invalid    134
Name: count, dtype: int64

Duplicate training rows: 0
Duplicate test rows: 0

Duplicate training Test_IDs: 0
Duplicate test Test_IDs: 0


In [9]:
BASE_FEATURES = [
    "Applied_Voltage_kV",
    "Load_Current_A",
    "Ambient_Temperature_C",
    "Test_Duration_min",
    "Sensor_S1",
    "Sensor_S2",
    "Sensor_S3",
    "Sensor_S4"
]

TARGET_REGRESSION = "Reference_Parameter"
TARGET_CLASSIFICATION = "Validity_Label"

print("Base features:")
for feature in BASE_FEATURES:
    print("-", feature)

Base features:
- Applied_Voltage_kV
- Load_Current_A
- Ambient_Temperature_C
- Test_Duration_min
- Sensor_S1
- Sensor_S2
- Sensor_S3
- Sensor_S4


In [10]:
train_clean = train.copy()
test_clean = test.copy()

SENSOR_COLUMNS = [
    "Sensor_S1",
    "Sensor_S2",
    "Sensor_S3",
    "Sensor_S4"
]

# Create missing-value indicators
for col in SENSOR_COLUMNS:
    train_clean[col + "_missing"] = train_clean[col].isnull().astype(int)
    test_clean[col + "_missing"] = test_clean[col].isnull().astype(int)

# Fill missing numeric values using TRAINING medians
for col in BASE_FEATURES:
    median_value = train_clean[col].median()

    train_clean[col] = train_clean[col].fillna(median_value)
    test_clean[col] = test_clean[col].fillna(median_value)

print("Missing values after cleaning:")

print("\nTraining:")
print(train_clean[BASE_FEATURES].isnull().sum())

print("\nTest:")
print(test_clean[BASE_FEATURES].isnull().sum())

Missing values after cleaning:

Training:
Applied_Voltage_kV       0
Load_Current_A           0
Ambient_Temperature_C    0
Test_Duration_min        0
Sensor_S1                0
Sensor_S2                0
Sensor_S3                0
Sensor_S4                0
dtype: int64

Test:
Applied_Voltage_kV       0
Load_Current_A           0
Ambient_Temperature_C    0
Test_Duration_min        0
Sensor_S1                0
Sensor_S2                0
Sensor_S3                0
Sensor_S4                0
dtype: int64


In [11]:
def add_features(df):
    df = df.copy()

    # Voltage × Current
    df["V_times_I"] = (
        df["Applied_Voltage_kV"] *
        df["Load_Current_A"]
    )

    # Sensor differences
    df["S1_minus_S2"] = df["Sensor_S1"] - df["Sensor_S2"]
    df["S1_minus_S3"] = df["Sensor_S1"] - df["Sensor_S3"]
    df["S2_minus_S3"] = df["Sensor_S2"] - df["Sensor_S3"]

    return df


train_clean = add_features(train_clean)
test_clean = add_features(test_clean)

print("Feature engineering completed.")

print("\nNew columns:")
print([
    "V_times_I",
    "S1_minus_S2",
    "S1_minus_S3",
    "S2_minus_S3"
])

Feature engineering completed.

New columns:
['V_times_I', 'S1_minus_S2', 'S1_minus_S3', 'S2_minus_S3']


In [12]:
REGRESSION_FEATURES = BASE_FEATURES + [
    "V_times_I"
]

CLASSIFICATION_FEATURES = BASE_FEATURES + [
    "V_times_I",
    "S1_minus_S2",
    "S1_minus_S3",
    "S2_minus_S3",
    "Sensor_S1_missing",
    "Sensor_S2_missing",
    "Sensor_S3_missing",
    "Sensor_S4_missing"
]

X_reg = train_clean[REGRESSION_FEATURES]
y_reg = train_clean[TARGET_REGRESSION]

X_class = train_clean[CLASSIFICATION_FEATURES]
y_class = train_clean[TARGET_CLASSIFICATION]

X_test_reg = test_clean[REGRESSION_FEATURES]
X_test_class = test_clean[CLASSIFICATION_FEATURES]

print("Regression features:", len(REGRESSION_FEATURES))
print("Classification features:", len(CLASSIFICATION_FEATURES))

print("Regression training shape:", X_reg.shape)
print("Classification training shape:", X_class.shape)

Regression features: 9
Classification features: 16
Regression training shape: (1000, 9)
Classification training shape: (1000, 16)


In [13]:
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestClassifier
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    f1_score,
    accuracy_score,
    classification_report
)

print("ML libraries loaded successfully!")

ML libraries loaded successfully!


In [14]:
regression_models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.10,
        max_depth=2,
        min_samples_split=20,
        min_samples_leaf=2,
        random_state=42
    ),

    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=42
    )
}

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

regression_results = []

for name, model in regression_models.items():

    scores = cross_val_score(
        model,
        X_reg,
        y_reg,
        cv=kf,
        scoring="neg_mean_absolute_error"
    )

    mae = -scores.mean()

    regression_results.append({
        "Model": name,
        "CV_MAE": mae
    })

regression_results = pd.DataFrame(regression_results)

regression_results = regression_results.sort_values(
    "CV_MAE"
)

print(regression_results)

                    Model    CV_MAE
3       Gradient Boosting  0.825199
2             Extra Trees  0.896257
4  Hist Gradient Boosting  0.925312
1           Random Forest  0.982814
0       Linear Regression  3.460417


In [15]:
best_regression_name = regression_results.iloc[0]["Model"]

best_regression_model = regression_models[best_regression_name]

print("BEST REGRESSION MODEL:")
print(best_regression_name)

best_regression_model.fit(
    X_reg,
    y_reg
)

BEST REGRESSION MODEL:
Gradient Boosting


GradientBoostingRegressor(max_depth=2, min_samples_leaf=2, min_samples_split=20,
                          n_estimators=500, random_state=42)

In [16]:
if hasattr(best_regression_model, "feature_importances_"):

    importance = pd.DataFrame({
        "Feature": REGRESSION_FEATURES,
        "Importance": best_regression_model.feature_importances_
    })

    importance = importance.sort_values(
        "Importance",
        ascending=False
    )

    print(importance)

else:
    print("Feature importance is not available for this model.")

                 Feature  Importance
1         Load_Current_A    0.834944
8              V_times_I    0.075412
2  Ambient_Temperature_C    0.041806
5              Sensor_S2    0.038168
3      Test_Duration_min    0.003359
6              Sensor_S3    0.002637
0     Applied_Voltage_kV    0.002261
4              Sensor_S1    0.001174
7              Sensor_S4    0.000239


In [17]:
test_reference_predictions = best_regression_model.predict(
    X_test_reg
)

print("Number of predictions:", len(test_reference_predictions))

print("\nFirst 10 predictions:")
print(test_reference_predictions[:10])

Number of predictions: 350

First 10 predictions:
[33.32136607 18.98799429 60.97143693 26.29608433 19.17052949 19.97251305
 18.43556029 15.11446525 33.66302534 18.06135728]


In [18]:
classification_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=700,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )
}

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

classification_results = []

for name, model in classification_models.items():

    scores = cross_val_score(
        model,
        X_class,
        y_class,
        cv=skf,
        scoring="f1_weighted"
    )

    f1 = scores.mean()

    classification_results.append({
        "Model": name,
        "CV_F1": f1
    })

classification_results = pd.DataFrame(
    classification_results
)

print(classification_results)

           Model     CV_F1
0  Random Forest  0.979422


In [19]:
best_classification_name = classification_results.iloc[0]["Model"]

best_classification_model = classification_models[
    best_classification_name
]

print("BEST CLASSIFICATION MODEL:")
print(best_classification_name)

best_classification_model.fit(
    X_class,
    y_class
)

BEST CLASSIFICATION MODEL:
Random Forest


RandomForestClassifier(class_weight='balanced', n_estimators=700, n_jobs=-1,
                       random_state=42)

In [20]:
if hasattr(best_classification_model, "feature_importances_"):

    class_importance = pd.DataFrame({
        "Feature": CLASSIFICATION_FEATURES,
        "Importance": best_classification_model.feature_importances_
    })

    class_importance = class_importance.sort_values(
        "Importance",
        ascending=False
    )

    print(class_importance)

                  Feature  Importance
9             S1_minus_S2    0.213975
10            S1_minus_S3    0.202947
11            S2_minus_S3    0.140403
5               Sensor_S2    0.070129
4               Sensor_S1    0.059583
6               Sensor_S3    0.054482
7               Sensor_S4    0.040704
3       Test_Duration_min    0.040157
0      Applied_Voltage_kV    0.037014
2   Ambient_Temperature_C    0.031860
1          Load_Current_A    0.031702
8               V_times_I    0.030304
14      Sensor_S3_missing    0.025986
12      Sensor_S1_missing    0.017420
13      Sensor_S2_missing    0.003117
15      Sensor_S4_missing    0.000217


In [21]:
test_validity_predictions = best_classification_model.predict(
    X_test_class
)

test_invalid_probability = (
    best_classification_model.predict_proba(
        X_test_class
    )[:, list(best_classification_model.classes_).index("Invalid")]
)

print("Validity predictions:", len(test_validity_predictions))

print("\nPrediction counts:")
print(pd.Series(test_validity_predictions).value_counts())

Validity predictions: 350

Prediction counts:
Valid      313
Invalid     37
Name: count, dtype: int64


In [22]:
attention = test_clean[
    [
        "Test_ID",
        "S1_minus_S2",
        "S1_minus_S3",
        "S2_minus_S3"
    ]
].copy()

# Sensor difference score
attention["sensor_difference_score"] = (
    attention["S1_minus_S2"].abs() +
    attention["S1_minus_S3"].abs() +
    attention["S2_minus_S3"].abs()
) / 3

# Missing sensor score
missing_columns = [
    "Sensor_S1_missing",
    "Sensor_S2_missing",
    "Sensor_S3_missing",
    "Sensor_S4_missing"
]

attention["missing_score"] = (
    test_clean[missing_columns].sum(axis=1) /
    len(missing_columns)
)

attention["invalid_probability"] = test_invalid_probability

# Normalize sensor difference
max_difference = attention["sensor_difference_score"].max()

if max_difference > 0:
    attention["normalized_difference"] = (
        attention["sensor_difference_score"] /
        max_difference
    )
else:
    attention["normalized_difference"] = 0

# Final attention score
attention["attention_score"] = (
    0.70 * attention["invalid_probability"] +
    0.20 * attention["normalized_difference"] +
    0.10 * attention["missing_score"]
)

top_3_attention = attention.sort_values(
    "attention_score",
    ascending=False
).head(3)

print("Top 3 Test IDs needing most attention:")
print(
    top_3_attention[
        ["Test_ID", "attention_score", "invalid_probability"]
    ]
)

Top 3 Test IDs needing most attention:
      Test_ID  attention_score  invalid_probability
257  TST-0258         0.867000             0.952857
262  TST-0142         0.863985             0.954286
7    TST-0178         0.862561             0.968571


In [24]:
submission = pd.DataFrame({
    "Test_ID": test_clean["Test_ID"],
    "Predicted_Reference_Parameter": test_reference_predictions,
    "Validity_Label": test_validity_predictions
})

print("Submission shape:", submission.shape)

print("\nFirst 10 rows:")
print(submission.head(10))

submission.to_csv(
    "TheFifthByte.csv",
    index=False
)

print("\nTheFifthByte.csv created successfully!")

Submission shape: (350, 3)

First 10 rows:
    Test_ID  Predicted_Reference_Parameter Validity_Label
0  TST-0278                      33.321366        Invalid
1  TST-0006                      18.987994          Valid
2  TST-0047                      60.971437          Valid
3  TST-0311                      26.296084          Valid
4  TST-0264                      19.170529        Invalid
5  TST-0169                      19.972513          Valid
6  TST-0269                      18.435560          Valid
7  TST-0178                      15.114465        Invalid
8  TST-0206                      33.663025          Valid
9  TST-0038                      18.061357          Valid

TheFifthByte.csv created successfully!


In [25]:
import json

summary = {
    "number_of_records": int(len(submission)),

    "number_valid": int(
        (submission["Validity_Label"] == "Valid").sum()
    ),

    "number_invalid": int(
        (submission["Validity_Label"] == "Invalid").sum()
    ),

    "minimum_predicted_reference_parameter": float(
        submission["Predicted_Reference_Parameter"].min()
    ),

    "maximum_predicted_reference_parameter": float(
        submission["Predicted_Reference_Parameter"].max()
    ),

    "average_predicted_reference_parameter": float(
        submission["Predicted_Reference_Parameter"].mean()
    ),

    "top_3_test_ids_needing_attention": (
        top_3_attention["Test_ID"].astype(str).tolist()
    ),

    "regression_model": best_regression_name,

    "regression_cv_mae": float(
        regression_results.iloc[0]["CV_MAE"]
    ),

    "classification_model": best_classification_name,

    "classification_cv_f1": float(
        classification_results.iloc[0]["CV_F1"]
    )
}

with open("summary.json", "w") as f:
    json.dump(summary, f, indent=4)

print(json.dumps(summary, indent=4))

print("\nsummary.json created successfully!")

{
    "number_of_records": 350,
    "number_valid": 313,
    "number_invalid": 37,
    "minimum_predicted_reference_parameter": 12.83295828772747,
    "maximum_predicted_reference_parameter": 60.97143692834163,
    "average_predicted_reference_parameter": 26.36223642528569,
    "top_3_test_ids_needing_attention": [
        "TST-0258",
        "TST-0142",
        "TST-0178"
    ],
    "regression_model": "Gradient Boosting",
    "regression_cv_mae": 0.8251987076927222,
    "classification_model": "Random Forest",
    "classification_cv_f1": 0.9794219469793648
}

summary.json created successfully!


In [26]:
print("========== FINAL CHECK ==========")

print(
    "Expected test rows: 350"
)

print(
    "Actual submission rows:",
    len(submission)
)

print(
    "Unique Test IDs:",
    submission["Test_ID"].nunique()
)

print(
    "Missing reference predictions:",
    submission["Predicted_Reference_Parameter"].isnull().sum()
)

print(
    "Missing validity predictions:",
    submission["Validity_Label"].isnull().sum()
)

print(
    "\nValidity counts:"
)

print(
    submission["Validity_Label"].value_counts()
)

print("\nReference Parameter statistics:")

print(
    submission["Predicted_Reference_Parameter"].describe()
)

if (
    len(submission) == 350
    and submission["Test_ID"].nunique() == 350
    and submission["Predicted_Reference_Parameter"].isnull().sum() == 0
    and submission["Validity_Label"].isnull().sum() == 0
):
    print("\n✅ FINAL CHECK PASSED!")
else:
    print("\n❌ SOMETHING NEEDS TO BE CHECKED!")

========== FINAL CHECK ==========
Expected test rows: 350
Actual submission rows: 350
Unique Test IDs: 350
Missing reference predictions: 0
Missing validity predictions: 0

Validity counts:
Validity_Label
Valid      313
Invalid     37
Name: count, dtype: int64

Reference Parameter statistics:
count    350.000000
mean      26.362236
std       10.181317
min       12.832958
25%       18.572177
50%       22.877197
75%       32.318961
max       60.971437
Name: Predicted_Reference_Parameter, dtype: float64

✅ FINAL CHECK PASSED!


In [27]:
regression_results.to_csv(
    "regression_model_comparison.csv",
    index=False
)

classification_results.to_csv(
    "classification_model_comparison.csv",
    index=False
)

if hasattr(best_regression_model, "feature_importances_"):
    importance.to_csv(
        "regression_feature_importance.csv",
        index=False
    )

if hasattr(best_classification_model, "feature_importances_"):
    class_importance.to_csv(
        "classification_feature_importance.csv",
        index=False
    )

print("All model results saved!")

All model results saved!
